In [1]:
import geomodel_toolbox
import importlib

importlib.reload(geomodel_toolbox)

Setting Backend To: AvailableBackends.numpy


<module 'geomodel_toolbox' from '/Users/ethanmeen/MSc-Thesis/GeoModel/geomodel_toolbox.py'>

In [16]:
import socket
print(socket.gethostname())

import sys
print(sys.executable)

c-002-007-035.client.tudelft.eduvpn.nl
/opt/anaconda3/envs/gempy310/bin/python


In [5]:
#  ====
# Inputs for creating strat model with gempy
#  ====

#xyz values are taken from gempy outputs
x_lims = (369702.8, 374398.9)
y_lims = (407778.9, 411690.2)
z_lims = (350, -2500)

resolution = [100, 100, 150]   # nx, ny, nz
 
strat_azimuth = 135 #deg
strat_dip = 10 #deg

#  ====
# Inputs for fracture map
#  ====

aperture = [50, 50] #[m] [MEAN, STD]
radius = [1000, 100] #[m] [MEAN, STD]
azimuth = [45, 10] #[deg] [MEAN, STD]

density = 5e-9 #[fractures/m3]

In [6]:
#geomodel_toolbox.create_strat_model(x_lims, y_lims, z_lims, resolution, strat_azimuth, strat_dip)

In [13]:
#Export well file for visualisation (separate from the geological model)
geomodel_toolbox.export_wells_as_dev()

# Import previously defined topo+strat model
lith_topo, x_array, y_array, z_array= geomodel_toolbox.import_strat_model(show_plot=False)

Created vertical_wells.dev


In [14]:
# Generate fracture model based on inputs
mask, df = geomodel_toolbox.generate_fracture_model(x_array, y_array, z_array, aperture, radius, azimuth, density, show_plot=False)

# Combine to get topo+strat+frac model
geomodel = lith_topo.copy()
geomodel[(lith_topo > 0) & mask] = lith_topo.max()+1

# Plot results
#geomodel_toolbox.plot_3d_model(geomodel, x_array, y_array, z_array)

In [8]:
# Define rock properties for easy access
ROCK_TYPES = {
    "M": {"poro": 0.2, "permx": 50},
    "B": {"poro": 0.02, "permx": 1e-2},
    "F": {"poro": 0.1, "permx": 1e3}
}

# Export to grid eclipse
grid = geomodel_toolbox.export_grdecl_from_geomodel(
    geomodel=geomodel,
    x_array=x_array,
    y_array=y_array,
    z_array=z_array,
    rock_types=ROCK_TYPES,
    out_file="grid.grdecl"
)

z_array_flat = -z_array[z_array < 0][::-1]

completions = geomodel_toolbox.export_wells_as_inc(
    wells_csv_path="gempy_inputs/borehole_data.csv",
    dem_path="subset_dem.tif",
    x_array=x_array,
    y_array=y_array,
    z_array_flat=z_array_flat,
    output_path="sch.inc",
)

sch.inc written with 37 completions.


In [10]:
import numpy as np

#print resolutions: 
print('x res:', np.round(x_array[1]-x_array[0], 1), 'm')
print('y res:', np.round(y_array[1]-y_array[0], 1), 'm')
print('z res:', np.round(z_array[1]-z_array[0], 1), 'm')

print(z_array_flat.shape)

x res: 47.0 m
y res: 39.1 m
z res: 19.0 m
(132,)
